# Online Retail — Exploratory Data Analysis

**Author:** Khaled Waleed  
**Dataset:** Online Retail II — UCI Machine Learning Repository  
**Period covered:** December 2009 – December 2011  
**Tool stack:** Python · Pandas · Matplotlib · Seaborn

---

## Project overview

This notebook is **Phase 1** of a three-phase E-Commerce RFM Analysis project.  
The goal here is to understand the raw dataset, clean it, and extract the key patterns  
that will inform the customer segmentation work in Phase 2.

### What we cover
1. [Load & Inspect](#1-load--inspect) — understand the structure and quality of the raw data  
2. [Data Cleaning](#2-data-cleaning) — remove noise and prepare a reliable working dataset  
3. [Exploratory Data Analysis](#3-exploratory-data-analysis) — uncover revenue trends, top markets, and customer behaviour  
4. [Export](#4-export) — save the clean dataset for Phase 2 (RFM Scoring)

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global plot style ───────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi'        : 150,
    'axes.titlesize'    : 14,
    'axes.labelsize'    : 12,
    'axes.titleweight'  : 'bold',
    'figure.facecolor'  : 'white',
})

# ── Brand colour palette (used consistently across all charts) ──
TEAL   = '#1D9E75'   # primary — growth / positive
CORAL  = '#D85A30'   # secondary — attention / warning
PURPLE = '#534AB7'   # geographic breakdowns
AMBER  = '#BA7517'   # product-level analysis
GRAY   = '#888780'   # neutral / inactive

---
## 1. Load & Inspect

Before touching any data, we need to understand what we're working with —  
the shape, column types, missing values, and overall date coverage.

In [ ]:
# Load the dataset
# Source: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
df = pd.read_csv('online_retail.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# Column types — check if InvoiceDate loaded as string (common issue)
print('── Column types ──')
print(df.dtypes)

print('\n── Sample rows ──')
df.head()

In [ ]:
# Missing values — Customer ID and Description are the usual suspects
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'Missing': missing, '% of total': missing_pct}).query('Missing > 0')

In [ ]:
# High-level dataset coverage
print(f'Date range   : {df["InvoiceDate"].min()}  →  {df["InvoiceDate"].max()}')
print(f'Customers    : {df["Customer ID"].nunique():,}')
print(f'Products     : {df["StockCode"].nunique():,}')
print(f'Countries    : {df["Country"].nunique()}')

**Observations from initial inspection:**

- `Customer ID` has ~243K missing values (~22% of rows) — these are guest/anonymous transactions and must be excluded before any customer-level analysis.
- `Description` has ~4K missing values — minor and won't affect revenue calculations.
- `InvoiceDate` loaded as `object` (string), so we'll need to parse it to `datetime`.
- The dataset spans roughly **2 years** (Dec 2009 – Dec 2011), which is solid for trend analysis.

---
## 2. Data Cleaning

We'll clean in a deliberate sequence — each step builds on the one before it.

| Step | Action | Reason |
|------|--------|--------|
| 2-A | Drop missing `Customer ID` | Can't link transaction to a customer |
| 2-B | Drop duplicate rows | Exact duplicates suggest data entry errors |
| 2-C | Isolate cancelled invoices | Keep for returns analysis, exclude from main dataset |
| 2-D | Remove invalid quantities & prices | Negative or zero values corrupt revenue totals |
| 2-E | Fix data types | `InvoiceDate` must be `datetime`; `Customer ID` as string |
| 2-F | Engineer new columns | `TotalPrice`, time breakdowns needed for EDA |

In [ ]:
original_len = len(df)

# 2-A: Drop rows with no Customer ID (anonymous / guest transactions)
df = df.dropna(subset=['Customer ID'])
dropped_no_id = original_len - len(df)

# 2-B: Remove fully duplicated rows
dups = df.duplicated().sum()
df = df.drop_duplicates()

# 2-C: Separate cancelled invoices (they start with 'C')
# We keep them in df_cancelled for the returns analysis later
cancelled_mask = df['Invoice'].astype(str).str.startswith('C')
df_cancelled   = df[cancelled_mask].copy()
df             = df[~cancelled_mask]

# 2-D: Remove rows where Quantity or Price make no business sense
invalid_qty   = (df['Quantity'] <= 0).sum()
invalid_price = (df['Price'] <= 0).sum()
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

print('── Cleaning log ──')
print(f'  Dropped (no Customer ID)  : {dropped_no_id:,}')
print(f'  Dropped (duplicates)      : {dups:,}')
print(f'  Isolated (cancellations)  : {cancelled_mask.sum():,}')
print(f'  Dropped (invalid qty)     : {invalid_qty:,}')
print(f'  Dropped (invalid price)   : {invalid_price:,}')
print(f'\n  Rows remaining : {len(df):,}')

In [ ]:
# 2-E: Fix data types

# Customer ID: convert float → int → string to avoid "12345.0" display issues
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# InvoiceDate: parse to proper datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print('Types after fix:')
print(df[['Customer ID', 'InvoiceDate']].dtypes)

In [ ]:
# 2-F: Engineer columns needed for the analysis

df['TotalPrice'] = df['Quantity'] * df['Price']     # revenue per line item
df['YearMonth']  = df['InvoiceDate'].dt.to_period('M')  # for monthly grouping
df['Month']      = df['InvoiceDate'].dt.month
df['Year']       = df['InvoiceDate'].dt.year
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()  # for weekday analysis

print(f'Clean dataset: {len(df):,} rows  |  {df["Customer ID"].nunique():,} unique customers')
df.head(3)

---
## 3. Exploratory Data Analysis

Now that the data is reliable, we explore six dimensions:

- **3-A** Monthly Revenue — how sales evolved over time
- **3-B** Top Countries — which export markets drive the most value
- **3-C** Top Products — which SKUs generate the most revenue
- **3-D** Orders by Day of Week — when do customers buy?
- **3-E** Revenue Distribution — how spending is spread across customers
- **3-F** Returns Rate — how significant are cancellations?

### 3-A · Monthly Revenue

A time-series view to spot seasonality and overall growth trajectory.

In [ ]:
monthly_rev = (
    df.groupby('YearMonth')['TotalPrice']
    .sum()
    .reset_index()
)
monthly_rev['YearMonth_str'] = monthly_rev['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(12, 4))

# Shaded area gives a sense of volume; line shows the trend
ax.fill_between(monthly_rev['YearMonth_str'], monthly_rev['TotalPrice'],
                alpha=0.15, color=TEAL)
ax.plot(monthly_rev['YearMonth_str'], monthly_rev['TotalPrice'],
        color=TEAL, linewidth=2.5, marker='o', markersize=5)

ax.set_title('Monthly Revenue')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('01_monthly_revenue.png')
plt.show()

**Insight:** Revenue shows a clear **Q4 seasonal spike** — November peaks every year, which aligns with Christmas gifting behaviour (the retailer sells mostly home décor and gifts). The 2011 peak is noticeably higher than 2010, suggesting year-on-year growth.

### 3-B · Top 10 Countries by Revenue (excl. UK)

The UK dominates total revenue, so we exclude it here to surface the export market dynamics.

In [ ]:
top_countries = (
    df[df['Country'] != 'United Kingdom']
    .groupby('Country')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    top_countries.index[::-1],
    top_countries.values[::-1],
    color=PURPLE, edgecolor='white', linewidth=0.5
)

# Add revenue labels at the end of each bar for easy reading
for bar in bars:
    ax.text(
        bar.get_width() + 500,
        bar.get_y() + bar.get_height() / 2,
        f'£{bar.get_width()/1000:.1f}K',
        va='center', fontsize=10
    )

ax.set_title('Top 10 Countries by Revenue (excl. UK)')
ax.set_xlabel('Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('02_top_countries.png')
plt.show()

**Insight:** **EIRE (Ireland)** leads international revenue by a significant margin, followed by the Netherlands and Germany. These three markets together likely represent the majority of non-UK revenue — worth tracking separately in future segmentation work.

### 3-C · Top 10 Products by Revenue

Identifying the highest-earning SKUs helps understand what drives the business — and what to protect.

In [ ]:
top_products = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    top_products.index[::-1],
    top_products.values[::-1],
    color=AMBER, edgecolor='white', linewidth=0.5
)

for bar in bars:
    ax.text(
        bar.get_width() + 200,
        bar.get_y() + bar.get_height() / 2,
        f'£{bar.get_width()/1000:.1f}K',
        va='center', fontsize=10
    )

ax.set_title('Top 10 Products by Revenue')
ax.set_xlabel('Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('03_top_products.png')
plt.show()

**Insight:** The **REGENCY CAKESTAND 3 TIER** is the top revenue-generating product — a high unit price item likely bought in bulk by cafés and event businesses. This hints that a segment of customers are B2B buyers, which will be an interesting angle in the RFM segmentation.

### 3-D · Orders by Day of Week

Understanding purchase timing helps optimise email campaigns, promotions, and stock replenishment.

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

orders_by_day = (
    df.groupby('DayOfWeek')['Invoice']
    .nunique()
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(8, 4))

# Highlight active weekdays vs the inactive weekend in GRAY
colors = [TEAL if d != 'Sunday' else GRAY for d in day_order]
ax.bar(orders_by_day.index, orders_by_day.values, color=colors, edgecolor='white')

ax.set_title('Number of Orders by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Unique Orders')

plt.tight_layout()
plt.savefig('04_orders_by_day.png')
plt.show()

**Insight:** Orders are almost exclusively placed on **weekdays**, with Thursday being the busiest day. Sundays show near-zero activity — likely a data artefact (the warehouse may not process orders on Sundays) rather than genuine zero demand.

### 3-E · Revenue Distribution per Customer

This histogram uses a log scale because a small number of customers generate disproportionately large revenue — a classic Pareto pattern. Log scale lets us see the full distribution without compressing the majority.

In [ ]:
customer_rev = df.groupby('Customer ID')['TotalPrice'].sum()

fig, ax = plt.subplots(figsize=(8, 4))

# log1p avoids log(0) errors and handles very small values gracefully
ax.hist(np.log1p(customer_rev), bins=50, color=CORAL, edgecolor='white', linewidth=0.4)

ax.set_title('Revenue Distribution per Customer (log scale)')
ax.set_xlabel('log(Revenue + 1)')
ax.set_ylabel('Number of Customers')

plt.tight_layout()
plt.savefig('05_revenue_distribution.png')
plt.show()

**Insight:** The distribution is **right-skewed** — most customers generate modest revenue, but a long tail of high-value customers exists. This validates the need for RFM segmentation in Phase 2: treating all customers the same would be a missed opportunity.

### 3-F · Returns Analysis

Cancelled invoices (prefix `'C'`) were separated during cleaning. Here we quantify the returns rate.

In [ ]:
total_invoices  = df['Invoice'].nunique()
total_cancelled = df_cancelled['Invoice'].nunique() if len(df_cancelled) else 0
returns_rate    = total_cancelled / (total_invoices + total_cancelled) * 100

print(f'Normal invoices    : {total_invoices:,}')
print(f'Cancelled invoices : {total_cancelled:,}')
print(f'Returns rate       : {returns_rate:.1f}%')

**Insight:** A **17.6% returns rate** is notably high for a retail business. This warrants further investigation in a dedicated returns analysis — possible causes include order errors, product quality issues, or bulk B2B buyers returning unsold stock. We'll flag this for the Power BI dashboard in Phase 3.

---
## 4. Export

Save the clean, enriched dataset to CSV — this becomes the input for Phase 2 (RFM Scoring).

In [ ]:
OUTPUT_PATH = 'retail_clean.csv'

df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved → {OUTPUT_PATH}')
print(f'Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Summary — Key Metrics

A quick reference of the headline numbers from this EDA.

In [ ]:
best_month_idx = monthly_rev['TotalPrice'].idxmax()
best_month_val = monthly_rev.loc[best_month_idx, 'TotalPrice']
best_month_lbl = monthly_rev.loc[best_month_idx, 'YearMonth_str']

metrics = {
    'Total Revenue'      : f"£{df['TotalPrice'].sum():,.0f}",
    'Total Orders'       : f"{df['Invoice'].nunique():,}",
    'Unique Customers'   : f"{df['Customer ID'].nunique():,}",
    'Avg Order Value'    : f"£{df.groupby('Invoice')['TotalPrice'].sum().mean():,.2f}",
    'Best Month'         : f"£{best_month_val:,.0f}  ({best_month_lbl})",
    'Top Country ex-UK'  : f"{top_countries.index[0]}  (£{top_countries.iloc[0]:,.0f})",
    'Top Product'        : top_products.index[0][:50],
    'Returns Rate'       : f"{returns_rate:.1f}%",
}

for label, value in metrics.items():
    print(f'  {label:<22}: {value}')

---
## Next Steps

**Phase 2 — RFM Scoring** (`rfm_phase2_scoring.ipynb`)

Using `retail_clean.csv`, we will:
1. Calculate **Recency**, **Frequency**, and **Monetary** values per customer
2. Score each metric on a 1–5 scale and compute a combined RFM score
3. Assign customers to segments: *Champions*, *Loyal*, *Potential*, *At-Risk*, *Lost*
4. Export the scored data for the Power BI dashboard